# Room Occupancy Estimation — Dataset Validation

This notebook validates the selected dataset before exploratory data analysis and preprocessing.

**Project task:** Predict `Room_Occupancy_Count` from environmental and motion sensor measurements.

**Machine-learning task:** Multiclass classification, because the target contains four occupancy classes: `0`, `1`, `2`, and `3`.


## 1. Dataset Source and Selection

The selected dataset is the **Room Occupancy Estimation** dataset from the UCI Machine Learning Repository.

- Public source: UCI Machine Learning Repository
- Dataset page: https://archive.ics.uci.edu/dataset/864/room+occupancy+estimation
- Target variable: `Room_Occupancy_Count`
- Data type: multivariate time-series sensor data
- Main predictors: temperature, light, sound, CO₂, CO₂ slope, and PIR motion measurements
- Intended project use: estimate how many people are present in the room

The dataset is suitable for the project because it provides a clear supervised-learning target, multiple numerical sensor variables, sufficient observations for model development, and realistic data-quality and temporal issues that can be investigated during EDA and preprocessing.


## 2. Load the Dataset


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("../data/raw/Occupancy_Estimation.csv")

assert DATA_PATH.exists(), f"Dataset not found at: {DATA_PATH.resolve()}"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()


## 3. Structure and Data Types

The following checks confirm the number of rows and columns, column names, and data types before further analysis.


In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumns:")
for col in df.columns:
    print("-", col)

print("\nData types:")
print(df.dtypes)


## 4. Target Validation

`Room_Occupancy_Count` is the prediction target.  
A valid classification target should contain a limited set of discrete class labels rather than continuous values.


In [ ]:
TARGET = "Room_Occupancy_Count"

assert TARGET in df.columns, f"{TARGET} is missing from the dataset."

target_counts = df[TARGET].value_counts().sort_index()
target_percent = (
    df[TARGET]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("Target classes:", sorted(df[TARGET].unique().tolist()))

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percent
})

target_summary


In [ ]:
expected_classes = {0, 1, 2, 3}
actual_classes = set(df[TARGET].dropna().unique())

assert actual_classes == expected_classes, (
    f"Unexpected target classes. Found: {sorted(actual_classes)}"
)

print("✅ Target validation passed: classes 0, 1, 2 and 3 are present.")


### Target Observation

The target is **imbalanced**. Class `0` (empty room) is much more common than the occupied classes.  
This does not invalidate the dataset, but it must be considered during model evaluation because accuracy alone may be misleading.


## 5. Missing Values and Duplicate Records

These checks identify two common data-quality problems before EDA.


In [ ]:
missing_total = int(df.isnull().sum().sum())
duplicate_total = int(df.duplicated().sum())

print("Total missing values:", missing_total)
print("Exact duplicate rows:", duplicate_total)

if missing_total == 0:
    print("✅ No missing values detected.")

if duplicate_total == 0:
    print("✅ No exact duplicate rows detected.")


## 6. Date and Time Validation

The observations are sequential sensor measurements.  
Checking the temporal structure is important because randomly splitting neighbouring observations between training and testing can create leakage-like similarity between the two sets.


In [ ]:
datetime_check = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    errors="coerce"
)

print("Unparseable date/time values:", datetime_check.isna().sum())
print("Earliest timestamp:", datetime_check.min())
print("Latest timestamp:", datetime_check.max())
print("Number of calendar dates:", datetime_check.dt.date.nunique())

time_differences = datetime_check.sort_values().diff().dropna()

print("\nMost common time intervals:")
print(time_differences.value_counts().head())


## 7. Sensor Feature Groups

The dataset contains several types of sensor measurements:

- `S1_Temp` to `S4_Temp`: temperature sensors
- `S1_Light` to `S4_Light`: light sensors
- `S1_Sound` to `S4_Sound`: sound sensor outputs
- `S5_CO2`: CO₂ measurement
- `S5_CO2_Slope`: change/slope in CO₂
- `S6_PIR` and `S7_PIR`: passive infrared motion indicators

These variables provide multiple signals that can be investigated for their relationship with room occupancy.


In [ ]:
sensor_columns = [
    c for c in df.columns
    if c not in ["Date", "Time", TARGET]
]

print("Number of predictor columns:", len(sensor_columns))
print("\nPredictor columns:")
print(sensor_columns)

print("\nNumeric predictor summary:")
df[sensor_columns].describe().T


## 8. Dataset Suitability Assessment

The dataset is suitable for this project because:

1. It is publicly available from a recognized machine-learning repository.
2. It has a clearly defined supervised-learning target.
3. The target contains four discrete classes, making the task a multiclass classification problem.
4. It contains multiple environmental and motion sensor predictors.
5. It has enough observations to support separate training and testing sets.
6. It contains no missing values or exact duplicate rows in the supplied CSV.
7. It includes realistic challenges such as class imbalance, correlated sensors, possible outliers, and temporal dependence.
8. The temporal structure can be handled explicitly during preprocessing to reduce leakage risk.

The class imbalance and temporal dependence do not make the dataset unsuitable; instead, they are important characteristics that need to be addressed and explained in later stages.


## 9. Validation Decision

**Decision: Proceed with the Room Occupancy Estimation dataset.**

The dataset is appropriate for PE1 because it supports problem definition, EDA, data-quality analysis, feature engineering, feature selection, train/test separation, and leakage prevention.

The next notebook, `02_eda.ipynb`, performs the detailed exploratory data analysis. The preprocessing decisions based on those findings are implemented in `03_preprocessing.ipynb`.


In [ ]:
print("=" * 60)
print("DATASET VALIDATION CHECK")
print("=" * 60)

assert df.shape[0] > 0
assert df.shape[1] == 19
assert TARGET in df.columns
assert set(df[TARGET].unique()) == {0, 1, 2, 3}
assert df.isnull().sum().sum() == 0
assert df.duplicated().sum() == 0

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Target classes:", sorted(df[TARGET].unique().tolist()))
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

print("\n✅ Dataset validation checks passed.")
